In [1]:
import sys

import itertools

import numpy as np
import pandas as pd
import scipy.stats as ss
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm as tqdm_auto
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import pytorch_lightning as pl

from IPython.display import clear_output

In [2]:
sys.path.append("../../predictor/model/")
import utrdata_cl as utrdata
from legnet import LegNetClassifier
from pl_regressor import RNARegressor

## Loading data

In [3]:
model_name = "legnet"
utr_type = "utr5"
device_id = 0 if utr_type == "utr5" else 1
seqsize = 50 if utr_type == "utr5" else 240

In [4]:
PATH_FROM = f"../../predictor/regression_multiple/{utr_type.upper()}_zscores_replicateagg.csv"
df = pd.read_csv(PATH_FROM)

In [5]:
n_splits = 10  # Remaining splits for the 'train' set

df['fold'] = df['fold'].map({'train': -1, 'val': n_splits - 2, 'test': n_splits - 1})

df_unsplit = df.loc[df['fold'] == -1, ['seq', 'fold']].copy()
df_unsplit['fold'] = df_unsplit['seq'].map(
    {seq: split_no
     for seq, split_no in
     zip(df_unsplit['seq'].drop_duplicates(),
         itertools.cycle(range(n_splits - 2)))
     }
)
df.loc[df_unsplit.index, 'fold'] = df_unsplit['fold']

In [6]:
fold_sizes = df['fold'].value_counts()
fold_size = fold_sizes.max()
fold_sizes

fold
0    10805
1    10805
2    10805
3    10805
4    10805
5    10805
6    10805
7    10805
8    10805
9    10800
Name: count, dtype: int64

In [7]:
num_classes = df["cell_type"].unique().shape[0]
num_classes

5

In [8]:
splits = dict(tuple(df.groupby('fold')))
splits[8].head()

,seq,cell_type,fold,1,2,3,4,mass_center,mass_center_mean,diff,zscore,mass_center_std
95,AAACAAATGTAGCAGGCACAGCGTGGGGTGGACAGTCAGCTGTCGG...,c1,8,68.449816,60.870873,47.590087,41.838243,2.287164,2.29667,-0.009506,-0.062351,0.152462
96,AAACAAATGTAGCAGGCACAGCGTGGGGTGGACAGTCAGCTGTCGG...,c17,8,72.812510,53.335530,34.751567,42.078913,2.227102,2.29667,-0.069567,-0.456294,0.152462
97,AAACAAATGTAGCAGGCACAGCGTGGGGTGGACAGTCAGCTGTCGG...,c2,8,71.853249,67.896625,57.267869,55.146572,2.379545,2.29667,0.082875,0.543581,0.152462
98,AAACAAATGTAGCAGGCACAGCGTGGGGTGGACAGTCAGCTGTCGG...,c4,8,77.312032,66.148593,64.039833,77.299136,2.496230,2.29667,0.199560,1.308917,0.152462
99,AAACAAATGTAGCAGGCACAGCGTGGGGTGGACAGTCAGCTGTCGG...,c6,8,86.538512,56.970286,42.241073,32.322653,2.093308,2.29667,-0.203362,-1.333853,0.152462


In [9]:
batch_size = 1024
train_size = fold_size * (n_splits - 2)
steps_per_epoch = max(1, int(train_size // batch_size))

# batch_per_epoch = 128  # None

# if batch_per_epoch is None:
#     # batch_per_epoch = int(np.ceil(splits["train"].shape[0] / batch_size))
#     batch_per_epoch = splits["train"].shape[0] // batch_size  # drop_last=True @ DataLoader

epochs = None

In [10]:
num_workers = 32

In [11]:
folds_arg = [{
    'train_folds': {j % n_splits for j in range(i, i + n_splits - 2)},
    'val_fold': (i - 2) % n_splits,
    'test_fold': (i - 1) % n_splits
} for i in range(n_splits)]
folds_arg

[{'train_folds': {0, 1, 2, 3, 4, 5, 6, 7}, 'val_fold': 8, 'test_fold': 9},
 {'train_folds': {1, 2, 3, 4, 5, 6, 7, 8}, 'val_fold': 9, 'test_fold': 0},
 {'train_folds': {2, 3, 4, 5, 6, 7, 8, 9}, 'val_fold': 0, 'test_fold': 1},
 {'train_folds': {0, 3, 4, 5, 6, 7, 8, 9}, 'val_fold': 1, 'test_fold': 2},
 {'train_folds': {0, 1, 4, 5, 6, 7, 8, 9}, 'val_fold': 2, 'test_fold': 3},
 {'train_folds': {0, 1, 2, 5, 6, 7, 8, 9}, 'val_fold': 3, 'test_fold': 4},
 {'train_folds': {0, 1, 2, 3, 6, 7, 8, 9}, 'val_fold': 4, 'test_fold': 5},
 {'train_folds': {0, 1, 2, 3, 4, 7, 8, 9}, 'val_fold': 5, 'test_fold': 6},
 {'train_folds': {0, 1, 2, 3, 4, 5, 8, 9}, 'val_fold': 6, 'test_fold': 7},
 {'train_folds': {0, 1, 2, 3, 4, 5, 6, 9}, 'val_fold': 7, 'test_fold': 8}]

In [12]:
checked = {
    "seed": [3],
    "folds": folds_arg,
    "features": [
        ("sequence", "positional", "conditions"),
    ],
    "augment_dict": [
        dict(
            extend_left=0,
            extend_right=0,
            shift_left=0,
            shift_right=0,
            revcomp=False,
        ),
    ],
    "epochs": [10],
}

In [13]:
def launch_model(
    seed: int,
    train_ds_kws: dict,
    val_ds_kws: dict,
    model_class,
    model_kws: dict,
    criterion_class,
    criterion_kws: dict,
    optimizer_class,
    optimizer_kws: dict,
    lr_scheduler_class,
    lr_scheduler_kws: dict,
    test_time_validation: bool,
    train_folds: set,
    val_fold: int,
    test_fold: int,
    initialize_weights: bool,
    epochs: int = epochs,
):
    pl.seed_everything(seed)

    # Creating Datasets
    train_set = utrdata.UTRData(
        df=pd.concat([splits[i] for i in train_folds]).sort_index().reset_index(drop=True),
        **train_ds_kws,
    )
    val_set = utrdata.UTRData(
        df=splits[val_fold].reset_index(drop=True),
        **val_ds_kws,
    )
    test_set = utrdata.UTRData(
        df=splits[test_fold].reset_index(drop=True),
        **val_ds_kws,
    )

    assert train_set.num_channels == val_set.num_channels
    try:
        div_factor = val_ds_kws["augment_kws"]["shift_left"] + \
                     val_ds_kws["augment_kws"]["shift_right"] + 1
    except KeyError:
        div_factor = 1

    # Creating DataLoaders
    dl_train = DataLoader(
        train_set,
        batch_size=batch_size,
        num_workers=num_workers,
        shuffle=True,
        drop_last=True
    )
    dl_val = DataLoader(
        val_set,
        batch_size=batch_size // div_factor,
        num_workers=num_workers,
        shuffle=False,
        drop_last=False
    )
    dl_test = DataLoader(
        test_set,
        batch_size=batch_size // div_factor,
        num_workers=num_workers,
        shuffle=False,
        drop_last=False
    )

    model = RNARegressor(
        model_class=model_class,
        model_kws=model_kws | dict(
            in_channels=train_set.num_channels
        ),
        criterion_class=criterion_class,
        criterion_kws=criterion_kws,
        optimizer_class=optimizer_class,
        optimizer_kws=optimizer_kws,
        lr_scheduler_class=lr_scheduler_class,
        lr_scheduler_kws=lr_scheduler_kws,
        test_time_validation=test_time_validation,
        initialize_weights=initialize_weights,
    )
    checkpoint_callback = pl.callbacks.ModelCheckpoint(
        dirpath=f"saved_models/{utr_type}/{model.model_name}",
        filename=f"fold={test_fold}-{{epoch}}-{{step}}",
        save_top_k=1,
        save_last=False,
        monitor="val_pearson_r_0",
        mode="max"
    )
    progressbar_callback = pl.callbacks.TQDMProgressBar(refresh_rate=0.5)

    logger = pl.loggers.tensorboard.TensorBoardLogger("tb_logs", name=model.model_name)
    trainer = pl.Trainer(
        callbacks=[checkpoint_callback, progressbar_callback],
        logger=logger,
        accelerator="gpu",
        devices=[device_id],
        deterministic=True,
        max_epochs=epochs,
        num_sanity_val_steps=0,
        # gradient_clip_val=1e-5,
        # gradient_clip_algorithm="norm",
    )
    trainer.fit(model=model, train_dataloaders=dl_train, val_dataloaders=dl_val)
    best_model = RNARegressor.load_from_checkpoint(checkpoint_callback.best_model_path)

    prediction = trainer.predict(model=best_model, dataloaders=dl_test)
    test_pred, test_real = zip(*prediction)
    test_pred = torch.concat(test_pred).numpy()
    test_real = torch.concat(test_real).numpy()
    test_df = splits[test_fold].copy()
    test_df["real_0"] = test_real[:, 0]
    test_df["real_1"] = test_real[:, 1]
    test_df["pred_0"] = test_pred[:, 0]
    test_df["pred_1"] = test_pred[:, 1]

    return trainer, test_df

In [14]:
results = []
for subset in itertools.product(
    *checked.values()
):
    PARAMS = dict(zip(checked.keys(), subset))
    AUGMENT_KEY = any(PARAMS["augment_dict"].values())
    AUGMENT_TEST_TIME = AUGMENT_KEY

    trainer_last, prediction_best_last = launch_model(
        seed=PARAMS["seed"],
        train_ds_kws=dict(
            construct_type=utr_type,
            features=PARAMS["features"],  # ("sequence", "conditions", "positional", "revcomp")
            augment=AUGMENT_KEY,
            augment_test_time=False,
            augment_kws=PARAMS["augment_dict"],
        ),
        val_ds_kws=dict(
            construct_type=utr_type,
            features=PARAMS["features"],  # ("sequence", "conditions", "positional", "revcomp")
            augment=False,
            augment_test_time=AUGMENT_TEST_TIME,
            augment_kws=PARAMS["augment_dict"],
        ),
        model_class=LegNetClassifier,
        model_kws=dict(
            seqsize=seqsize,
            in_channels=4,  # IS REPLACED
            ks=3,
            out_channels=2,
            conv_sizes=(128, 64, 64, 32, 32),
            mapper_size=256,
            linear_sizes=None,
            use_max_pooling=False,
            final_activation=nn.Identity
        ),
        criterion_class=nn.MSELoss,
        criterion_kws=dict(),
        optimizer_class=torch.optim.AdamW,
        optimizer_kws=dict(
            # lr=0.01,
            weight_decay=0.1,
        ),
        lr_scheduler_class=torch.optim.lr_scheduler.OneCycleLR,
        lr_scheduler_kws=dict(
            max_lr=0.01,
            steps_per_epoch=steps_per_epoch,
            epochs=PARAMS["epochs"],
            pct_start=0.3,
            three_phase=False,
            cycle_momentum=True,
        ),
        test_time_validation=AUGMENT_TEST_TIME,
        **PARAMS["folds"],
        initialize_weights=True,
        epochs=PARAMS["epochs"],
    )
    results.append((subset, prediction_best_last))

Global seed set to 3
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name      | Type             | Params
-----------------------------------------------
0 | model     | LegNetClassifier | 268 K 
1 | criterion | MSELoss          | 0     
-----------------------------------------------
268 K     Trainable params
0         Non-trainable params
268 K     Total params
1.073     Total estimated model params size (MB)


Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

...

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

`Trainer.fit` stopped: `max_epochs=10` reached.
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting: 0it [00:00, ?it/s]

## Validation

In [15]:
full_result = pd.concat([r for _, r in results])
full_result

,seq,cell_type,fold,1,2,3,4,mass_center,mass_center_mean,diff,zscore,mass_center_std,real_0,real_1,pred_0,pred_1
205,AAACCGAGGCCAGAGTGTCCCCGTGGGCCGAGCGCACTTTTTTCTT...,c1,9,76.676858,77.280305,71.076231,71.867445,2.465254,2.421816,0.043438,0.460575,0.094312,0.043438,2.465254,0.004116,2.540994
206,AAACCGAGGCCAGAGTGTCCCCGTGGGCCGAGCGCACTTTTTTCTT...,c17,9,79.028058,87.374265,63.447506,53.471756,2.322471,2.421816,-0.099345,-1.053367,0.094312,-0.099345,2.322471,-0.000911,2.526515
207,AAACCGAGGCCAGAGTGTCCCCGTGGGCCGAGCGCACTTTTTTCTT...,c2,9,74.376089,76.078916,74.830791,67.914497,2.464814,2.421816,0.042998,0.455914,0.094312,0.042998,2.464814,0.028963,2.550840
208,AAACCGAGGCCAGAGTGTCCCCGTGGGCCGAGCGCACTTTTTTCTT...,c4,9,57.917050,74.023513,66.633378,66.185816,2.532891,2.421816,0.111074,1.177732,0.094312,0.111074,2.532891,-0.003860,2.491256
209,AAACCGAGGCCAGAGTGTCCCCGTGGGCCGAGCGCACTTTTTTCTT...,c6,9,111.268903,68.903514,67.302508,74.007200,2.323651,2.421816,-0.098165,-1.040854,0.094312,-0.098165,2.323651,-0.036071,2.480810
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107955,TTTGTGTGTCCCGTCGTAGCGGGGGACCGCGTGTGTGCTTGCTTCT...,c1,8,41.063813,45.451728,43.822668,39.705082,2.483224,2.710146,-0.226922,-1.206529,0.188078,-0.226922,2.483224,-0.025883,2.576374
107956,TTTGTGTGTCCCGTCGTAGCGGGGGACCGCGTGTGTGCTTGCTTCT...,c17,8,25.435698,48.080785,69.751576,90.704895,2.964751,2.710146,0.254605,1.353718,0.188078,0.254605,2.964751,0.094508,2.785096
107957,TTTGTGTGTCCCGTCGTAGCGGGGGACCGCGTGTGTGCTTGCTTCT...,c2,8,39.493947,38.811318,44.620602,46.650347,2.580431,2.710146,-0.129715,-0.689685,0.188078,-0.129715,2.580431,-0.023055,2.572107
107958,TTTGTGTGTCCCGTCGTAGCGGGGGACCGCGTGTGTGCTTGCTTCT...,c4,8,19.089273,25.384751,33.779972,40.007828,2.800819,2.710146,0.090673,0.482104,0.188078,0.090673,2.800819,-0.025095,2.594785


In [16]:
full_result.to_csv(f"results/{utr_type}_{model_name}_crossval.csv", index=False)

In [17]:
full_result.groupby('cell_type')[['mass_center', 'pred_1']].corr().unstack(-1)[('mass_center', 'pred_1')]

cell_type
c1     0.786669
c17    0.754684
c2     0.791176
c4     0.652270
c6     0.798295
Name: (mass_center, pred_1), dtype: float64

In [18]:
full_result.groupby('seq')[['mass_center', 'pred_1']].mean().corr().stack()[('mass_center', 'pred_1')]

0.8312814917891771

In [19]:
full_result.groupby(['cell_type', 'fold'])[['mass_center', 'pred_1']].corr().unstack(-1)[('mass_center', 'pred_1')].unstack(0)

cell_type,c1,c17,c2,c4,c6
fold,,,,,
0,0.774318,0.740443,0.778125,0.624154,0.789843
1,0.790861,0.770320,0.802858,0.669522,0.809727
2,0.798205,0.758550,0.791424,0.652760,0.806320
3,0.793353,0.747045,0.790934,0.670345,0.806144
4,0.755710,0.735901,0.758301,0.644233,0.783716
5,0.792262,0.773862,0.792216,0.663930,0.801643
6,0.793238,0.762106,0.799463,0.644208,0.800338
7,0.793542,0.751626,0.801777,0.635595,0.808171
8,0.800855,0.766052,0.809899,0.670229,0.797893


In [20]:
full_result.groupby(['seq', 'fold'])[['mass_center', 'pred_1']].mean().groupby(level='fold').corr().unstack(-1)[('mass_center', 'pred_1')]

fold
0    0.818075
1    0.843093
2    0.837294
3    0.833641
4    0.811171
5    0.839530
6    0.834074
7    0.839060
8    0.835818
9    0.823007
Name: (mass_center, pred_1), dtype: float64

---

In [21]:
x = torch.arange(30).reshape(2, 3, 5)
y = torch.arange(30).reshape(2, 3, 5) + 100
torch.cat([x, y], dim=1).shape

torch.Size([2, 6, 5])